Notebook 2b: Dictionary lookup, and BERT with dictionary candidates

This notebook uses a word list in two ways. First as a non-GenAI baseline: a plain dictionary lookup. A word list is built from the gold
text of the BLN600 excerpts outside the sample, and every damaged word is replaced by the list entry whose
spelling is closest to its damaged form. There is no context and no model. It is the pure "confusion" side of
"context beats confusion" (Evershed and Fitch, 2014): BERT and the LLM add the context, so this baseline shows
what the context is worth. Second, the same word list gives BERT extra candidate words (the second half of
this notebook), because BERT on its own only proposes words from the context.

**Input.** The frozen damaged dataset `data/processed/corrupted_v2.jsonl` (hash checked against
`runs/corrupted_v2.sha256`), BLN600 in `data/raw/BLN600/`, `configs/lexicon.yaml`, `configs/bert_repair_lex.yaml`, and the fine-tuned model `models/bert_ft_v1/`
(notebook 2).

**Output.** The stored predictions in `runs/preds/lexicon_<version>.jsonl` and
`runs/preds/bert_rerank_<version>.jsonl` (versions with `-x<n>`), scored in notebook 4 together
with the other methods.

`SPLIT` chooses the rows. The test split stays closed until `ALLOW_TEST` is set and `frozen: true` is set in
the method's config. The lookup has nothing to tune; for BERT with dictionary candidates, the number of
candidates and λ are chosen on dev only.

In [1]:
import inspect

import pandas as pd

from blnrepair.data import ROOT, load_config, load_gold
from blnrepair.freeze import load_frozen
from blnrepair.lexicon import LexiconLookup, build_lexicon, lexicon_version, load_lexicon_config
from blnrepair.preds import load_preds, pred_path, run_method, slot_exact
from blnrepair.slots import build_slots, slot_view

pd.set_option("display.max_colwidth", None)
config, lex_cfg = load_config(), load_lexicon_config()
rows = load_frozen("v2")  # stops if the file hash does not match runs/corrupted_v2.sha256

SPLIT, ALLOW_TEST = "test", True
if SPLIT == "test":
    assert ALLOW_TEST and lex_cfg["frozen"], "the test split is closed: set ALLOW_TEST and freeze configs/lexicon.yaml"

The word list

The list holds every word of the 443 excerpts that are not in the sample, so no test or dev sentence
contributes a word. A word is taken without its edge punctuation, with the same rule as the slot view
(inner hyphens and apostrophes stay, so `Market-place` is one entry), and case is kept. Each entry keeps its
count, which breaks ties below.

In [2]:
sample_docs = {r["doc_id"] for r in rows}
gold = load_gold(ROOT / config["paths"]["raw_dir"])
counts = build_lexicon(gold, sample_docs)
print(f"{len(gold) - len(sample_docs)} excerpts outside the sample, {len(counts):,} distinct words, "
      f"{sum(counts.values()):,} words in total")

443 excerpts outside the sample, 16,469 distinct words, 213,237 words in total


The lookup

For each slot with visible letters, the damaged form is compared with every entry of the list by edit
distance (insertions, deletions and substitutions of single characters, case-sensitive, the same measure
BERT's reranking uses). The closest entry wins; among equally close entries, the most frequent one. A dropped
word `⟨?⟩` has no letters to look up, so the lookup leaves it empty: the word stays missing and counts as
wrong.

In [3]:
print(inspect.getsource(LexiconLookup.nearest))

    def nearest(self, core):
        """(entry, edit distance, number of entries at that distance) for one damaged core."""
        dist = cdist([core], self.words, scorer=Levenshtein.distance, workers=-1)[0]
        best = int(dist.min())
        i = int(dist.argmin())  # first minimum = the most frequent of the closest entries
        return self.words[i], best, int((dist == best).sum())



Running the lookup

The lookup runs on every row of the split that has slots, through the same prediction store as the other
methods. It takes a few seconds for a whole split.

In [4]:
lookup = LexiconLookup(counts)
version = lexicon_version(lex_cfg, SPLIT)
split_rows = [r for r in rows if r["split"] == SPLIT]
added = run_method(split_rows, lookup, "lexicon", version, "dictionary lookup", allow_test=ALLOW_TEST)
stored = load_preds(pred_path("lexicon", version))
print(f"lexicon {version}: {len(stored)} rows stored ({len(added)} new in this run)")

lexicon v1-test: 750 rows stored (0 new in this run)


Examples from dev

A few dev repairs at the one-word level, with the damaged form, the lookup's answer and the gold word. The
examples always come from dev, whatever `SPLIT` is, so no test result is shown here. The share of exact
words per level is a sanity check, not a result; the scores are in notebook 4.

In [5]:
dev_rows = [r for r in rows if r["split"] == "dev" and r["k"]]
dev_preds = {(p["id"], p["severity"]): p for p in load_preds(pred_path("lexicon", lexicon_version(lex_cfg, "dev")))}
one_word = [r for r in dev_rows if r["severity"] == "1w"][:8]
pd.DataFrame({"damaged": [r["ops_per_word"][0]["out"] for r in one_word],
              "lookup": [dev_preds[(r["id"], "1w")]["pred_words"][0] for r in one_word],
              "gold": [r["ops_per_word"][0]["gold"] for r in one_word]})

,damaged,lookup,gold
0,"£1,",1,"£2,"
1,Raiiy-ststion,manifestation,Railway-station
2,Wiiliani,William,William
3,"NIaruet-plece,",Dorset-place,"Market-place,"
4,"l8,",18,"18,"
5,Suney,money,Sunday
6,Hainos,Haines,Haines
7,"Wrighb,",Wright,"Wright,"


In [6]:
pd.Series({level: sum(slot_exact(dev_preds[(r["id"], level)]["pred_words"], r)
                      for r in dev_rows if r["severity"] == level) / sum(r["severity"] == level for r in dev_rows)
           for level in ["1w", "10", "25", "50", "75"]}, name="exact words, dev (sanity check)").round(3)

1w    0.450
10    0.463
25    0.529
50    0.610
75    0.588
Name: exact words, dev (sanity check), dtype: float64

BERT with dictionary candidates

BERT (notebook 2) builds its candidate words from the context alone: the 5 most likely pieces per mask. The
damaged letters only re-sort that list, so a word BERT does not propose can never be chosen, however close
its letters are. On dev, for `⟨Wrighb⟩` BERT's candidates were Smith, White and andooll, for `⟨Suney⟩` Saturday,
Wednesday and Monday. Here the word list adds the entries closest in spelling to each damaged part (by edit
distance, then frequency) to BERT's own candidates, and every candidate gets the same score as before: BERT's
mean log-probability per piece plus λ times the letter similarity. The model and everything else stay as in
notebook 2, and BERT still never sees damaged text in training. A split slot (`Pol1ce-coart`) is looked up
part by part, among the word list's parts. A dictionary word's BERT score is read from the forward pass whose
number of masks equals its number of pieces, which is run anyway; a word of more than 3 pieces is skipped, as
for BERT's own candidates.

In [7]:
import torch
import transformers
from rapidfuzz.distance import Levenshtein
from transformers import AutoModelForMaskedLM, AutoTokenizer

from blnrepair.bert_repair import BertReranker, load_repair_config, repair_version
from blnrepair.slots import slot_core, splice

transformers.logging.set_verbosity_error()
transformers.logging.disable_progress_bar()

print(inspect.getsource(BertReranker.dictionary_candidates))

/Users/simonm/UniRGB_local/GenAI/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


    def dictionary_candidates(self, logps_by_m, damaged):
        """The lex_n word-list parts closest to the damaged part, with their BERT score read from the
        pass whose mask count equals their number of pieces (skipped above max_masks or with [UNK])."""
        out = {}
        for word in self.lex.closest(damaged, self.lex_n):
            ids = self.tok(word, add_special_tokens=False)["input_ids"]
            if not ids or len(ids) > self.max_masks or self.tok.unk_token_id in ids:
                continue
            lp = logps_by_m[len(ids) - 1]
            out[word] = sum(lp[i, t].item() for i, t in enumerate(ids)) / len(ids)
        return out



Choosing the number of candidates and λ on dev

The grid repairs all 100 dev rows for 5, 10, 20 and 50 dictionary candidates per part and λ = 1, 2, 4, 8, 16,
32 and 64, plus λ = 128 for 5 candidates (the first grid stopped at 16, where the scores were still rising, so
it was extended until they flattened). It runs on the Apple GPU (MPS) where there is one: on 40 dev rows the MPS run chose exactly the same words
as the CPU and as the stored run of notebook 2, at twice the speed. The table uses the same letter-level scores
as the λ grid in notebook 2 (means over all dev rows). The setting is chosen by the same rule as there: the
best exact-word rate at a point where the Fact Recovery Rate has reached its plateau. BERT without dictionary
candidates (λ = 8, notebook 2) and the dictionary lookup are listed for reference. The dev split is used for this
choice only; the test split never chooses.

In [8]:
lex_bert_cfg = load_repair_config(ROOT / "configs" / "bert_repair_lex.yaml")
device = "mps" if torch.backends.mps.is_available() else "cpu"
source = ROOT / lex_bert_cfg["model"]
model, tokenizer = AutoModelForMaskedLM.from_pretrained(source), AutoTokenizer.from_pretrained(source)
GRID = [(n, lam) for n in [5, 10, 20, 50] for lam in [1, 2, 4, 8, 16, 32, 64]] + [(5, 128)]
dev_all = [r for r in rows if r["split"] == "dev"]  # the grid is always on dev: the test split never chooses a setting
grid_versions = {(n, lam): repair_version({**lex_bert_cfg, "lam": lam, "lex_n": n}, "dev") for n, lam in GRID}
for (n, lam), version in grid_versions.items():
    if len(load_preds(pred_path("bert_rerank", version))) < len(dev_rows):
        reranker = BertReranker(model, tokenizer, lam=lam, beam=lex_bert_cfg["beam"], top_n=lex_bert_cfg["top_n"],
                                device=device, lexicon=counts, lex_n=n)
        run_method(dev_all, reranker, "bert_rerank", version, lex_bert_cfg["model"])  # allow_test stays False

In [9]:
def letter_scores(row, words):
    # exact words, fact recovery and span CER before / after for one repaired row (as in notebook 2)
    plans = sorted(row["ops_per_word"], key=lambda p: p["idx"])
    s, e = row["span_start"], row["span_end"]
    span = {"gold_tokens": row["gold_tokens"][s:e], "ops_per_word": [{**p, "idx": p["idx"] - s} for p in plans]}
    gold_span = " ".join(row["gold_tokens"][s:e])
    cer = lambda text: Levenshtein.distance(text, gold_span) / len(gold_span)
    facts = [slot_core(w) == slot_core(p["gold"]) for w, p in zip(words, plans) if p["idx"] in row["fact_idx_in_span"]]
    return {"exact": slot_exact(words, row), "fact recovery": sum(facts) / len(facts),
            "CER before": cer(splice(span, [p["out"] for p in plans])), "CER after": cer(splice(span, words))}


by_key = {(r["id"], r["severity"]): r for r in rows}
runs = {("BERT, no dictionary (notebook 2)", 8): ("bert_rerank", repair_version(load_repair_config(), "dev")),
        ("dictionary lookup", "-"): ("lexicon", lexicon_version(lex_cfg, "dev")),
        **{(f"BERT + {n} dictionary words", lam): ("bert_rerank", v) for (n, lam), v in grid_versions.items()}}
grid = pd.DataFrame([{"method": m, "λ": lam, "level": p["severity"], **letter_scores(by_key[(p["id"], p["severity"])], p["pred_words"])}
                     for (m, lam), path in runs.items() for p in load_preds(pred_path(*path))])
grid["repair gain"] = grid["CER before"] - grid["CER after"]
grid.groupby(["method", "λ"], sort=False)[["exact", "fact recovery", "CER after", "repair gain"]].mean().round(3)

exact  fact recovery  CER after  \
method                           λ                                      
BERT, no dictionary (notebook 2) 8    0.267          0.126      0.572   
dictionary lookup                -    0.528          0.426      0.211   
BERT + 5 dictionary words        1    0.164          0.042      0.756   
                                 2    0.227          0.118      0.655   
                                 4    0.286          0.175      0.537   
                                 8    0.416          0.275      0.408   
                                 16   0.617          0.499      0.188   
                                 32   0.619          0.537      0.155   
                                 64   0.619          0.560      0.151   
BERT + 10 dictionary words       1    0.164          0.042      0.756   
                                 2    0.227          0.118      0.655   
                                 4    0.285          0.176      0.538   
                                 8    0.420          0.278      0.398   
                                 16   0.606          0.480      0.192   
                                 32   0.610          0.517      0.159   
                                 64   0.613          0.558      0.152   
BERT + 20 dictionary words       1    0.164          0.042      0.756   
                                 2    0.228          0.122      0.654   
                                 4    0.286          0.180      0.535   
                                 8    0.420          0.281      0.397   
                                 16   0.612          0.494      0.189   
                                 32   0.615          0.526      0.158   
                                 64   0.592          0.553      0.159   
BERT + 50 dictionary words       1    0.164          0.042      0.756   
                                 2    0.228          0.122      0.655   
                                 4    0.286          0.180      0.535   
                                 8    0.420          0.282      0.394   
                                 16   0.609          0.489      0.191   
                                 32   0.628          0.553      0.155   
                                 64   0.606          0.580      0.155   
BERT + 5 dictionary words        128  0.603          0.558      0.156   

                                      repair gain  
method                           λ                 
BERT, no dictionary (notebook 2) 8         -0.269  
dictionary lookup                -          0.092  
BERT + 5 dictionary words        1         -0.453  
                                 2         -0.352  
                                 4         -0.234  
                                 8         -0.106  
                                 16         0.115  
                                 32         0.148  
                                 64         0.152  
BERT + 10 dictionary words       1         -0.453  
                                 2         -0.352  
                                 4         -0.235  
                                 8         -0.096  
                                 16         0.111  
                                 32         0.144  
                                 64         0.150  
BERT + 20 dictionary words       1         -0.453  
                                 2         -0.351  
                                 4         -0.232  
                                 8         -0.094  
                                 16         0.114  
                                 32         0.145  
                                 64         0.144  
BERT + 50 dictionary words       1         -0.453  
                                 2         -0.352  
                                 4         -0.232  
                                 8         -0.091  
                                 16         0.112  
                                 32         0.148  
                

Exact words by setting and level (dev):

In [10]:
grid.pivot_table(index=["method", "λ"], columns="level", values="exact", aggfunc="mean", sort=False)[["1w", "10", "25", "50", "75"]].round(3)

level                                   1w     10     25     50     75
method                           λ                                    
BERT, no dictionary (notebook 2) 8    0.25  0.407  0.274  0.224  0.182
dictionary lookup                -    0.45  0.463  0.529  0.610  0.588
BERT + 5 dictionary words        8    0.45  0.496  0.420  0.396  0.320
                                 1    0.10  0.263  0.197  0.146  0.112
                                 2    0.30  0.330  0.202  0.174  0.128
                                 4    0.35  0.361  0.282  0.238  0.199
                                 16   0.65  0.666  0.582  0.606  0.581
                                 32   0.60  0.649  0.605  0.636  0.604
                                 64   0.65  0.656  0.598  0.615  0.578
                                 128  0.65  0.643  0.576  0.591  0.554
BERT + 10 dictionary words       8    0.45  0.521  0.414  0.394  0.321
                                 1    0.10  0.263  0.197  0.146  0.112
                                 2    0.30  0.330  0.202  0.174  0.128
                                 4    0.35  0.361  0.282  0.238  0.196
                                 16   0.65  0.628  0.584  0.586  0.582
                                 32   0.60  0.633  0.591  0.629  0.600
                                 64   0.65  0.656  0.592  0.607  0.562
BERT + 20 dictionary words       8    0.45  0.521  0.424  0.394  0.313
                                 1    0.10  0.263  0.197  0.146  0.112
                                 2    0.30  0.330  0.206  0.174  0.128
                                 4    0.35  0.361  0.286  0.238  0.196
                                 16   0.65  0.628  0.587  0.604  0.590
                                 32   0.60  0.633  0.592  0.631  0.618
                                 64   0.65  0.639  0.554  0.574  0.541
BERT + 50 dictionary words       8    0.45  0.521  0.428  0.394  0.309
                                 1    0.10  0.263  0.197  0.146  0.112
                                 2    0.30  0.330  0.206  0.174  0.128
                                 4    0.35  0.361  0.286  0.238  0.196
                                 16   0.65  0.612  0.595  0.604  0.581
                                 32   0.65  0.649  0.598  0.635  0.609
                                 64   0.70  0.656  0.561  0.577  0.537

The chosen setting

**5 dictionary candidates per part and λ = 64** (`configs/bert_repair_lex.yaml`). The number of candidates
matters little and without a pattern: at λ = 16 to 64, 5, 10, 20 and 50 are within about 0.03 of each other,
and the best one changes from one λ to the next (the highest exact-word rate, 0.628, is 50 candidates at λ = 32;
5 candidates reach 0.619). On 100 dev rows, 0.01 is about 10 slots, so these differences are noise and the
smallest number is kept. λ
matters a lot. Exact words rise up to λ = 16 and stay flat from there (0.617, 0.619, 0.619 at 16, 32, 64), and
the Fact Recovery Rate keeps rising up to λ = 64 and is flat from 64 to 128 (0.560, 0.558), while exact words
drop again at 128. So λ = 64 is the best exact-word rate at the point where the Fact Recovery Rate has reached
its plateau. At this λ the letters decide in most cases and BERT's reading of the context mainly breaks near
ties and fills the dropped words; how much that adds over the plain lookup is part of the comparison in
notebook 4. The chosen setting is run on `SPLIT` below; its dev run is the grid point above.

In [11]:
lex_bert_version = repair_version(lex_bert_cfg, SPLIT)
if SPLIT == "test":
    assert ALLOW_TEST and lex_bert_cfg["frozen"], "the test split is closed: set ALLOW_TEST and freeze configs/bert_repair_lex.yaml"
reranker = BertReranker(model, tokenizer, lam=lex_bert_cfg["lam"], beam=lex_bert_cfg["beam"], top_n=lex_bert_cfg["top_n"],
                        device=device, lexicon=counts, lex_n=lex_bert_cfg["lex_n"])
added = run_method(split_rows, reranker, "bert_rerank", lex_bert_version, lex_bert_cfg["model"], allow_test=ALLOW_TEST)
stored = load_preds(pred_path("bert_rerank", lex_bert_version))
print(f"bert_rerank {lex_bert_version}: {len(stored)} rows stored ({len(added)} new in this run)")
pd.DataFrame([{"level": p["severity"], "seconds": p["seconds"]} for p in stored]).groupby("level")["seconds"].agg(["count", "mean"]).round(1)

bert_rerank ftv1-l64-b5-n10-x5-test: 750 rows stored (750 new in this run)


,count,mean
level,,
10,150,0.2
1w,150,0.1
25,150,0.4
50,150,0.9
75,150,1.3
